An arXiv research assistant has two different jobs: turn documents into a searchable collection, and use that collection to answer questions. The first job can run on a schedule; the second must respond while a user waits. Most of the engineering in this course concerns the boundary between those jobs: identity, freshness, failures, resource limits, and evidence that the answer is supported.

This is an independent teaching reconstruction of Jam With AI's [Week 1 release](https://github.com/jamwithai/production-agentic-rag-course/tree/week1.0). The release provides the implementation to inspect; the explanations, small experiments, and review exercises here develop the engineering ideas behind it. They do not reproduce the original recorded course. Familiarity with Python functions, HTTP requests, and basic SQL is enough to begin.

The objective is to explain why the components exist, predict what breaks when a dependency fails, and collect evidence from a complete Docker deployment. The Python cells below require only the standard library. The Docker lab is part of completing the lesson, but its results must be collected on a machine able to run the stack; the commands are instructions, not a record of an executed deployment.

## The request and its evidence

A representative question is: “How does retrieval help a language model answer questions about recent papers?” A useful answer needs passages from the right papers, an association between those passages and their sources, and a generator that can use the passages. A running model alone cannot supply those guarantees.

**Retrieval-augmented generation** conditions generation on information selected from an external collection. The original [RAG paper](https://arxiv.org/abs/2005.11401) combined a learned retriever and generator. This project uses a broader engineering pattern: separately ingest documents, retrieve passages, construct a prompt, and call a model. It does not reproduce that paper's joint training procedure.

The offline path is arXiv → parsing → durable storage → search indexing. The online path is question → retrieval → context assembly → generation → response. “Offline” here means outside the user request, not disconnected from the internet. Separating these paths prevents PDF parsing from becoming part of every answer's latency.

```{mermaid}
flowchart TD
    I[Scheduled ingestion] --> P[PostgreSQL records]
    P --> S[OpenSearch index]
    Q[User request] --> A[FastAPI application]
    A --> S
    S --> C[Retrieved context]
    C --> L[Ollama generation]
    L --> R[Answer and source references]
```

The diagram describes the course's eventual data flow, rather than claiming that every arrow is implemented in Week 1. Week 1 establishes the processes and connections; later weeks fill in ingestion, search, and generation.

## The responsibility of each service

PostgreSQL stores records with identity and relational constraints. OpenSearch stores a representation organized for retrieval. The search index is a **derived view**: ideally it can be rebuilt from durable records and retained document artifacts. Maintaining two stores creates an eventual-consistency problem. A committed record does not automatically imply a searchable passage; Week 2 studies that gap.

FastAPI owns the request contract and coordinates dependencies. Ollama serves a local model. Airflow schedules work and records task state. Docker Compose describes how these processes are built, configured, connected, and given storage. Containers make the deployment repeatable; they do not make an application correct or automatically suitable for production.

The release's [`compose.yml`](https://github.com/jamwithai/production-agentic-rag-course/blob/week1.0/compose.yml), [`src/main.py`](https://github.com/jamwithai/production-agentic-rag-course/blob/week1.0/src/main.py), and [`src/routers/ping.py`](https://github.com/jamwithai/production-agentic-rag-course/blob/week1.0/src/routers/ping.py) reveal three layers of the same request: the processes and ports, application startup, and the behavior exposed to a caller. Connecting these layers makes configuration understandable in terms of an observable request.

There is a cost to every separate service: memory, startup time, network calls, logs, credentials, and another failure mode. The course uses a broad stack to expose those concerns. A smaller application might reasonably combine responsibilities or defer Airflow until ingestion needs scheduling and retries. Architecture should answer requirements, not reward the highest tool count.

## A hostname belongs to a network context

Inside the API container, `localhost` refers to that container. It does not refer to the PostgreSQL container or to the host laptop. Compose service names such as `postgres` are resolved on the Compose network; a host-side notebook normally reaches a published port through `localhost` instead.

The same dependency therefore has two addresses. For example, a host-side client may use `localhost:9200`, while the API uses `opensearch:9200`. Published ports provide host access; services on the same Docker network can communicate using the container port without publishing it. A connection failure can consequently be a naming mistake even when every service is running.

Volumes are a separate concern. A named volume preserves database files when a container is replaced, but preservation is not a backup. It does not protect against accidental deletion, bad migrations, or application-level corruption. Image versions, configuration, database schema, model files, and data volumes together determine reproducibility.

## Running, ready, and useful are different observations

A process can exist before it accepts connections. It can accept connections before its schema is initialized. It can answer a health endpoint while the corpus is empty. Each boundary calls for a different observation:

| Observation | What it establishes | What it does not establish |
|---|---|---|
| Process is running | The process has not exited | It can serve a request |
| Dependency probe succeeds | A particular dependency operation works | All application paths work |
| API readiness succeeds | The chosen readiness contract is satisfied | Answers are grounded or relevant |
| A known-answer query passes | One functional scenario works | General quality or load capacity |

Docker's [startup-order documentation](https://docs.docker.com/compose/how-tos/startup-order/) distinguishes starting a container from waiting for a health check. `depends_on` with `service_healthy` can gate initial startup, but runtime failures still require application behavior: timeouts, bounded retries, and a response indicating which capability is unavailable.

**Experiment: model capability-based readiness.** The following is a proposed policy, not a claim about the release's health handler. 

1. Predict whether an unavailable Ollama server should prevent a search-only request.
2. Compare that prediction with a generation request.


In [ ]:
def readiness(required, observed):
    missing = sorted(name for name in required if not observed.get(name, False))
    return {"ready": not missing, "missing": missing}

services = {"postgres": True, "opensearch": True, "ollama": False}
search = readiness({"postgres", "opensearch"}, services)
answer = readiness({"postgres", "opensearch", "ollama"}, services)
assert search["ready"] is True
assert answer == {"ready": False, "missing": ["ollama"]}
print("Search capability:", search)
print("Answer capability:", answer)


Reference output from isolated Python execution:

```text
Search capability: {'ready': True, 'missing': []}
Answer capability: {'ready': False, 'missing': ['ollama']}
```

The experiment separates a policy decision from a probe implementation. Requiring every service for every request couples unrelated capabilities and can turn an optional dependency's failure into an application-wide outage. Conversely, reporting healthy while silently skipping an essential dependency misleads callers. A useful health response names the capability and its actual requirements.

The Week 1 health handler probes the database and Ollama, while a `/ping` handler checks basic HTTP reachability. Neither response proves that OpenSearch contains the right papers. The meaning of “healthy” is limited to the dependencies and operations the handler actually checks.

## Latency budgets are architectural constraints

Suppose retrieval takes 120 ms, embedding a question takes 80 ms, prompt preparation takes 10 ms, and generation takes 1,500 ms. A sequential request takes about 1,710 ms before additional transport overhead. Optimizing prompt preparation by half saves 5 ms. Optimizing generation by a third saves 500 ms. The numbers here are illustrative inputs, not measurements of this project.

For sequential stages, $T_{\mathrm{request}}=\sum_i T_i$. Independent operations executed concurrently have a different critical path. Concurrency can reduce waiting, but it cannot remove a dependency: generation cannot use retrieval results before they exist. It also cannot turn CPU-bound parsing into free work.

**Experiment: calculate where a proposed optimization can matter.** Comparing a partial improvement with the complete elimination of a stage gives an upper bound on its potential contribution to request latency.


In [ ]:
stages_ms = {"embedding": 80, "retrieval": 120, "prompt": 10, "generation": 1500}
baseline = sum(stages_ms.values())
for stage, latency in stages_ms.items():
    improved = baseline - latency / 2
    print(f"Halve {stage:10s}: {improved:7.1f} ms; speedup {baseline/improved:.3f}x")
assert baseline == 1710
print("Upper bound if prompt work vanished:", round(baseline / (baseline - 10), 3))


Reference output from isolated Python execution:

```text
Halve embedding :  1670.0 ms; speedup 1.024x
Halve retrieval :  1650.0 ms; speedup 1.036x
Halve prompt    :  1705.0 ms; speedup 1.003x
Halve generation:   960.0 ms; speedup 1.781x
Upper bound if prompt work vanished: 1.006
```

A component's percentage improvement is not the request's percentage improvement. The calculation also omits queueing, cold model loading, retries, and concurrency. Later measurements should include those effects. A benchmark reporting only a warm, single request can miss the behavior users experience when several requests arrive together.

## Docker lab: establish the complete deployment

1. Run this section later on the capable laptop. Use the matching release for each week; old notebooks are not guaranteed to work against the Week 7 source. Keep each checkout separate and run only one course stack at a time: published ports and explicit container names can conflict even with different Compose project names.

    ```bash
    git clone --branch week1.0 --single-branch \
      https://github.com/jamwithai/production-agentic-rag-course.git rag-week1
    cd rag-week1
    cp .env.example .env
    git rev-parse HEAD
    docker compose --project-name rag-week1 config --quiet
    docker compose --project-name rag-week1 config --services
    docker compose --project-name rag-week1 up --build -d
    docker compose --project-name rag-week1 ps
    curl --fail-with-body http://localhost:8000/ping
    curl --fail-with-body http://localhost:8000/health
    curl --fail-with-body http://localhost:9200/_cluster/health
    curl --fail-with-body http://localhost:11434/api/tags
    docker stats --no-stream
    ```

2. Copy the environment file only on the first setup; preserve later configuration edits. The Dockerfile installs application dependencies inside the image, so installing the whole course environment on the host is not necessary merely to start the stack. Record the commit, Docker version, platform, resource allocation, image versions, and the health-response bodies. HTTP 200 alone is insufficient if a response body reports a degraded service. Ollama responding with no installed models proves service reachability, not generation readiness.

3. Open the Airflow interface on port 8080 and OpenSearch Dashboards on port 5601. Compare the full running service list with the Compose definition. A successful lab establishes a baseline for every declared service, rather than selectively running only the API. If startup fails, inspect the relevant dependency first:

    ```bash
    docker compose --project-name rag-week1 logs --tail=100 api postgres opensearch airflow ollama
    ```

4. **Failure injection.** Stop Ollama, repeat `/ping` and `/health`, then restart it. Predict both responses before observing them. A functioning ping alongside a degraded health response demonstrates why liveness and dependency readiness answer different questions.

    ```bash
    docker compose --project-name rag-week1 stop ollama
    curl --fail-with-body http://localhost:8000/ping
    curl --fail-with-body http://localhost:8000/health
    docker compose --project-name rag-week1 start ollama
    ```

5. The anticipated result is that HTTP reachability survives while generation-related health degrades. If the API exits or startup coupling prevents recovery, investigate that behavior rather than rewriting the expectation to call it success. Before changing weeks, use `docker compose --project-name rag-week1 down` to remove this stack's containers and network while keeping named volumes. Do not add `--volumes` as routine troubleshooting: that erases state instead of explaining the failure.

## Engineering review and evidence to retain

1. Explain why PostgreSQL and OpenSearch both appear in the design. Then describe how the system could recover after losing only its search index. The explanation should identify which artifacts are sufficient to rebuild, rather than assuming the answer is “restart Docker.”

2. Choose one endpoint and write its dependency contract, timeout budget, and degraded behavior. A strong answer distinguishes a failed connection from an empty corpus and explains what the caller can safely retry. Review the Compose configuration for development conveniences—published databases, disabled search security, default credentials—and describe what must change before exposing it beyond a local environment.

3. Retain the deployment observations and failure-injection results as a short engineering record. The completion criterion is the ability to connect an observed response to the responsible process, network address, and dependency contract. That reasoning is the foundation for debugging ingestion in Week 2.

## Sources and scope

- [Week 1 source and notebook](https://github.com/jamwithai/production-agentic-rag-course/tree/week1.0), including `compose.yml`, application startup, and health routes. The project distributes its code under the [MIT license](https://github.com/jamwithai/production-agentic-rag-course/blob/week1.0/LICENSE).
- [Docker Compose startup order](https://docs.docker.com/compose/how-tos/startup-order/): what dependency conditions establish.
- [Lewis et al., Retrieval-Augmented Generation](https://arxiv.org/abs/2005.11401): the original research formulation, distinct from this application's orchestration pattern.

The small experiments are executable models of specific ideas. They do not validate Docker networking, service recovery, retrieval quality, or production capacity; the integration lab supplies those additional observations.
